# Single-Qubit Cliffords to NMR Pulses

This notebook enumerates all 24 single-qubit Clifford gates, writes each one as an NMR-friendly pulse sequence, and validates the full set with QuTiP.

Conventions:
- Sequences are written in time order from left to right.
- `Free(phi)` denotes free evolution that accumulates a z rotation by `phi`, so `Free(phi) == Rz(phi)`.
- All validations are done up to global phase.


## The 24 single-qubit Cliffords and pulse sequences

| # | Clifford | Pulse sequence |
|---|---|---|
| 1 | `I` | `I` |
| 2 | `Rx(+pi/2)` | `Rx(+pi/2)` |
| 3 | `Rx(-pi/2)` | `Rx(-pi/2)` |
| 4 | `Ry(+pi/2)` | `Ry(+pi/2)` |
| 5 | `Ry(-pi/2)` | `Ry(-pi/2)` |
| 6 | `Rz(+pi/2)` | `Free(+pi/2)` |
| 7 | `Rz(-pi/2)` | `Free(-pi/2)` |
| 8 | `Rx(pi)` | `Rx(pi)` |
| 9 | `Ry(pi)` | `Ry(pi)` |
| 10 | `Rz(pi)` | `Free(pi)` |
| 11 | `R_(+x,+y,+z)(+2pi/3)` | `Rx(+pi/2) -> Free(+pi/2)` |
| 12 | `R_(+x,-y,-z)(+2pi/3)` | `Rx(+pi/2) -> Free(-pi/2)` |
| 13 | `R_(+x,+y,-z)(+2pi/3)` | `Rx(+pi/2) -> Ry(+pi/2)` |
| 14 | `R_(+x,-y,+z)(+2pi/3)` | `Rx(+pi/2) -> Ry(-pi/2)` |
| 15 | `R_(+x,+y,+z)(-2pi/3)` | `Rx(-pi/2) -> Ry(-pi/2)` |
| 16 | `R_(+x,-y,-z)(-2pi/3)` | `Rx(-pi/2) -> Ry(+pi/2)` |
| 17 | `R_(+x,+y,-z)(-2pi/3)` | `Rx(-pi/2) -> Free(+pi/2)` |
| 18 | `R_(+x,-y,+z)(-2pi/3)` | `Rx(-pi/2) -> Free(-pi/2)` |
| 19 | `R_(+x,+y,0)(pi)` | `Rx(pi) -> Free(+pi/2)` |
| 20 | `R_(+x,-y,0)(pi)` | `Rx(pi) -> Free(-pi/2)` |
| 21 | `R_(+x,0,+z)(pi)` | `Rx(pi) -> Ry(-pi/2)` |
| 22 | `R_(+x,0,-z)(pi)` | `Rx(pi) -> Ry(+pi/2)` |
| 23 | `R_(0,+y,+z)(pi)` | `Rx(+pi/2) -> Free(pi)` |
| 24 | `R_(0,+y,-z)(pi)` | `Rx(+pi/2) -> Ry(pi)` |


In [ ]:
import numpy as np
import qutip as qt
from itertools import permutations, product
from IPython.display import Markdown, display

pi = np.pi
sx, sy, sz = qt.sigmax(), qt.sigmay(), qt.sigmaz()
I2 = qt.qeye(2)

def Rx(theta):
    return (-1j * theta / 2 * sx).expm()

def Ry(theta):
    return (-1j * theta / 2 * sy).expm()

def Rz(theta):
    return (-1j * theta / 2 * sz).expm()

def Free(phi):
    return Rz(phi)

PRIMITIVES = {"Rx": Rx, "Ry": Ry, "Free": Free}

def format_sequence(sequence):
    if not sequence:
        return "I"
    pieces = []
    for op, theta in sequence:
        if np.isclose(theta, pi):
            angle = "pi"
        elif np.isclose(theta, -pi):
            angle = "-pi"
        elif np.isclose(theta, pi / 2):
            angle = "+pi/2"
        elif np.isclose(theta, -pi / 2):
            angle = "-pi/2"
        else:
            angle = f"{theta:.6g}"
        pieces.append(f"{op}({angle})")
    return " -> ".join(pieces)

def apply_sequence(sequence):
    U = I2
    for op, theta in sequence:
        U = PRIMITIVES[op](theta) * U
    return U

def canonicalize_global_phase(U):
    flat = U.full().flatten()
    idx = next((i for i, z in enumerate(flat) if abs(z) > 1e-10), None)
    if idx is None:
        return U
    return U * np.exp(-1j * np.angle(flat[idx]))

def unitary_key(U):
    arr = canonicalize_global_phase(U).full()
    real = tuple(np.round(arr.real, 10).flatten())
    imag = tuple(np.round(arr.imag, 10).flatten())
    return real + imag

def same_unitary(U, V, tol=1e-10):
    return np.allclose(canonicalize_global_phase(U).full(), canonicalize_global_phase(V).full(), atol=tol)

def axis_angle_unitary(axis, angle):
    axis = np.array(axis, dtype=float)
    if np.isclose(angle, 0.0):
        return I2
    axis = axis / np.linalg.norm(axis)
    generator = axis[0] * sx + axis[1] * sy + axis[2] * sz
    return (-1j * angle / 2 * generator).expm()

def bloch_action(U):
    basis = [sx, sy, sz]
    R = np.zeros((3, 3), dtype=int)
    for i, P in enumerate(basis):
        conj = U * P * U.dag()
        coeffs = [((Q * conj).tr() / 2).real for Q in basis]
        R[:, i] = np.rint(coeffs).astype(int)
    return R

def is_proper_signed_permutation(R):
    return (
        np.array_equal(np.abs(R).sum(axis=0), np.ones(3, dtype=int))
        and np.array_equal(np.abs(R).sum(axis=1), np.ones(3, dtype=int))
        and round(np.linalg.det(R)) == 1
    )

def target_rotations():
    mats = []
    for perm in permutations(range(3)):
        P = np.zeros((3, 3), dtype=int)
        for col, row in enumerate(perm):
            P[row, col] = 1
        for signs in product((-1, 1), repeat=3):
            R = np.diag(signs) @ P
            if round(np.linalg.det(R)) == 1:
                mats.append(R)
    unique = []
    seen = set()
    for R in mats:
        key = tuple(R.flatten())
        if key not in seen:
            seen.add(key)
            unique.append(R)
    return unique


In [ ]:
CLIFFORDS = [
    {"name": "I", "axis": (0, 0, 1), "angle": 0.0, "sequence": []},
    {"name": "Rx(+pi/2)", "axis": (1, 0, 0), "angle": pi / 2, "sequence": [("Rx", pi / 2)]},
    {"name": "Rx(-pi/2)", "axis": (1, 0, 0), "angle": -pi / 2, "sequence": [("Rx", -pi / 2)]},
    {"name": "Ry(+pi/2)", "axis": (0, 1, 0), "angle": pi / 2, "sequence": [("Ry", pi / 2)]},
    {"name": "Ry(-pi/2)", "axis": (0, 1, 0), "angle": -pi / 2, "sequence": [("Ry", -pi / 2)]},
    {"name": "Rz(+pi/2)", "axis": (0, 0, 1), "angle": pi / 2, "sequence": [("Free", pi / 2)]},
    {"name": "Rz(-pi/2)", "axis": (0, 0, 1), "angle": -pi / 2, "sequence": [("Free", -pi / 2)]},
    {"name": "Rx(pi)", "axis": (1, 0, 0), "angle": pi, "sequence": [("Rx", pi)]},
    {"name": "Ry(pi)", "axis": (0, 1, 0), "angle": pi, "sequence": [("Ry", pi)]},
    {"name": "Rz(pi)", "axis": (0, 0, 1), "angle": pi, "sequence": [("Free", pi)]},
    {"name": "R_(+x,+y,+z)(+2pi/3)", "axis": (1, 1, 1), "angle": 2 * pi / 3, "sequence": [("Rx", pi / 2), ("Free", pi / 2)]},
    {"name": "R_(+x,-y,-z)(+2pi/3)", "axis": (1, -1, -1), "angle": 2 * pi / 3, "sequence": [("Rx", pi / 2), ("Free", -pi / 2)]},
    {"name": "R_(+x,+y,-z)(+2pi/3)", "axis": (1, 1, -1), "angle": 2 * pi / 3, "sequence": [("Rx", pi / 2), ("Ry", pi / 2)]},
    {"name": "R_(+x,-y,+z)(+2pi/3)", "axis": (1, -1, 1), "angle": 2 * pi / 3, "sequence": [("Rx", pi / 2), ("Ry", -pi / 2)]},
    {"name": "R_(+x,+y,+z)(-2pi/3)", "axis": (1, 1, 1), "angle": -2 * pi / 3, "sequence": [("Rx", -pi / 2), ("Ry", -pi / 2)]},
    {"name": "R_(+x,-y,-z)(-2pi/3)", "axis": (1, -1, -1), "angle": -2 * pi / 3, "sequence": [("Rx", -pi / 2), ("Ry", pi / 2)]},
    {"name": "R_(+x,+y,-z)(-2pi/3)", "axis": (1, 1, -1), "angle": -2 * pi / 3, "sequence": [("Rx", -pi / 2), ("Free", pi / 2)]},
    {"name": "R_(+x,-y,+z)(-2pi/3)", "axis": (1, -1, 1), "angle": -2 * pi / 3, "sequence": [("Rx", -pi / 2), ("Free", -pi / 2)]},
    {"name": "R_(+x,+y,0)(pi)", "axis": (1, 1, 0), "angle": pi, "sequence": [("Rx", pi), ("Free", pi / 2)]},
    {"name": "R_(+x,-y,0)(pi)", "axis": (1, -1, 0), "angle": pi, "sequence": [("Rx", pi), ("Free", -pi / 2)]},
    {"name": "R_(+x,0,+z)(pi)", "axis": (1, 0, 1), "angle": pi, "sequence": [("Rx", pi), ("Ry", -pi / 2)]},
    {"name": "R_(+x,0,-z)(pi)", "axis": (1, 0, -1), "angle": pi, "sequence": [("Rx", pi), ("Ry", pi / 2)]},
    {"name": "R_(0,+y,+z)(pi)", "axis": (0, 1, 1), "angle": pi, "sequence": [("Rx", pi / 2), ("Free", pi)]},
    {"name": "R_(0,+y,-z)(pi)", "axis": (0, 1, -1), "angle": pi, "sequence": [("Rx", pi / 2), ("Ry", pi)]},
]


In [ ]:
unitaries = [apply_sequence(item["sequence"]) for item in CLIFFORDS]

assert len(CLIFFORDS) == 24
assert len({unitary_key(U) for U in unitaries}) == 24

for item, U in zip(CLIFFORDS, unitaries):
    expected = axis_angle_unitary(item["axis"], item["angle"])
    assert same_unitary(U, expected), item["name"]

rotations = [bloch_action(U) for U in unitaries]
assert all(is_proper_signed_permutation(R) for R in rotations)
assert {tuple(R.flatten()) for R in rotations} == {tuple(R.flatten()) for R in target_rotations()}

lookup = {unitary_key(U) for U in unitaries}
for Ua in unitaries:
    for Ub in unitaries:
        assert unitary_key(Ua * Ub) in lookup

print("Validated 24 unique single-qubit Cliffords with QuTiP.")
print("Each pulse sequence matches its declared axis-angle gate up to global phase.")
print("Each Bloch action is a proper signed permutation of {X, Y, Z}.")
print("The 24-element set is closed under composition.")


In [ ]:
lines = ["| # | Clifford | Pulse sequence |", "|---|---|---|"]
for idx, item in enumerate(CLIFFORDS, start=1):
    lines.append(f"| {idx} | `{item['name']}` | `{format_sequence(item['sequence'])}` |")
display(Markdown("\n".join(lines)))
